# Neural Networks: Regularization, Dropout, Batch Norm & Early Stopping

**Agenda**
- Overfitting recap: why train/val gap happens
- Prepping categorical data + standardization
- Baseline model + right loss function
- Fix 1: L2 Regularization
- Fix 2: Dropout
- Fix 3: Batch Normalization
- Fix 4: Early Stopping
- AI in the loop: how automation extends this

**Flow of the class:**
```
Baseline  -->  Diagnose Overfitting  -->  L2 Reg  -->  Dropout  -->  BatchNorm  -->  Early Stopping
```



## The Story: QuickCart

- Meera, data scientist at QuickCart (grocery delivery startup)
- Problem: predict at order-time if a delivery will miss its promised window, so ops can re-route early
- Already knows logistic regression — this needs a NN because features interact (heavy order + rain + new rider ≠ sum of parts)
- We follow her build, step by step, hitting the same wall she does each time



## Recap: Overfitting

**Talking points:**
- Analogy: student who memorizes practice-paper answers instead of the method — great on the same paper, falls apart on a new one
- Network does the same if it has enough capacity: fits real pattern *and* training-set noise
- Tell-tale sign: **training accuracy >> validation accuracy**

*(Everything today is a way to keep that gap small.)*

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/241/978/original/ChatGPT_Image_Sep_23__2026__05_33_06_PM.png?1790165002" height=500>

## Prepping the Data: Categories -> Numbers

**Talking points:**
- QuickCart data: numeric (distance, weight) + categorical (zone, delivery mode, rider tier)
- NN needs numbers only -> use **target encoding**: replace category with historical average of the outcome for that category

$$TE(c) = \frac{1}{n_c} \sum_{i:\ category_i = c} y_i$$

- $n_c$ = training rows in category $c$ &nbsp; | &nbsp; $y_i$ = label for row $i$ (1 = late, 0 = on time)
- Caution: very small $n_c$ -> noisy average -> usually smoothed toward overall mean (mention only, don't derive)



### Quiz 1
`on_foot` category: 40 training rows, 6 were late.

What's the target-encoded value?
- A) 0.06  
- B) 0.15  
- C) 0.40  
- D) 6.00

**Answer: B) 0.15** — $6/40 = 0.15$.

## Why Scale Matters: Standardization

**Talking points:**
- Distance in km vs weight in grams = comparing height in metres to age in days — not on the same footing
- Unscaled features -> stretched, lopsided loss surface -> gradient descent zig-zags
- Fix: z-score every feature

$$z = \frac{x - \mu}{\sigma}$$

- $x$ = raw value, $\mu$ = feature mean, $\sigma$ = feature std dev (both from training data)
- Result: more symmetric "bowl" -> faster, stabler convergence

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/241/984/original/ChatGPT_Image_Sep_23__2026__05_36_30_PM.png?1790165201" height=500>

## Baseline Model + Right Loss

**Talking points:**
- Few dense layers, final neuron: sigmoid -> outputs probability $\hat{y} \in [0,1]$
- True label $y \in \{0,1\}$ -> mismatch between continuous prediction and hard label
- Correct loss: **Binary Cross-Entropy**

$$L = -\frac{1}{m}\sum_{i=1}^{m}\Big[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\Big]$$

- $m$ = number of examples, $y_i$ = true label, $\hat{y}_i$ = predicted probability
- Heavily punishes confident wrong predictions



### Quiz 2
Why is MSE a weak choice here vs Binary Cross-Entropy?
- A) MSE can't run with sigmoid outputs
- B) MSE gives a much gentler gradient on confidently-wrong probability predictions, slowing learning
- C) MSE only works with 3+ classes
- D) They always give identical gradients

**Answer: B)** — BCE keeps a strong gradient exactly when the model is confidently wrong; MSE flattens out there.

Why not simply use MSE?
Mean Squared Error can be used in some binary setups, but BCE is generally better suited to binary probability prediction. With a Sigmoid output, MSE can produce a weaker learning signal for confidently incorrect predictions because of the Sigmoid derivative.

## Reading the Training Curve

| Metric | Value |
|---|---|
| Train Accuracy | 71% |
| Val Accuracy | 63% |

- 8-point gap -> model has capacity to fit noise (rare routes, one-off rider mistakes, freak weather days)
- Meera needs to hold the network back — without losing what it's learned correctly



## Fix 1 — L2 Regularization

**Analogy:** every weight = a volume knob. Overfitting = a few knobs cranked way up for tiny quirks. L2 = a spring quietly pulling every knob back toward zero unless the data really insists otherwise.

$$L_{reg} = L + \frac{\lambda}{2m}\sum (W^L)^2 \qquad\qquad W^L := W^L - \eta\Big(dW^L + \frac{\lambda}{m}W^L\Big)$$

- $\lambda$ = regularization strength (hyperparameter) | $m$ = # examples | $W^L$ = layer $L$'s weights
- $\eta$ = learning rate | $dW^L$ = original gradient (before regularization)
- Aka **weight decay** — every update shrinks weights a little, in addition to reducing error



### Quiz 3
Meera sets $\lambda$ way too high. Most likely effect?
- A) Faster training, nothing else changes
- B) Model overfits even more
- C) Weight-decay term dominates -> weights collapse toward 0 -> model underfits
- D) No effect on updates

**Answer: C)** — too strong a spring overpowers the real learning signal.

### Ask AI
*After class:* ask an AI assistant — difference between **L1** vs **L2** regularization, why L1 can zero out weights entirely while L2 only shrinks them, and the geometric reason (diamond vs circle constraint region).

## Fix 2 — Dropout

**Analogy:** a team that only ever plays its 5 stars collapses if one is out. A coach who randomly benches players in *practice* forces everyone to become independently capable -> more robust team.

- On each training step, randomly "bench" a fraction $r$ of neurons in a layer

$$d^l_j = \begin{cases} 1 & \text{prob } (1-r) \\ 0 & \text{prob } r \end{cases} \qquad a^l_{dropped} = \frac{a^l \circ d^l}{1-r}$$

- $r$ = dropout rate
- $\circ$ = elementwise multiply
- division by $(1-r)$ keeps expected activation level unchanged (**inverted dropout**)

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/241/988/original/ChatGPT_Image_Sep_23__2026__05_41_16_PM.png?1790165487" height=500>

### Quiz 4
Layer has 256 neurons, $r = 0.25$. Expected neurons dropped?
- A) 25  
- B) 64  
- C) 192  
- D) 256

**Answer: B) 64** — $0.25 \times 256 = 64$.

## Dropout: Train vs Test

- Masking happens **only during training**
- At test time: all neurons active, no masking, no further adjustment needed
- Why it just works: inverted scaling during training already calibrated the network for "full capacity" behaviour

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/243/433/original/img.png?1790683512" width="800">

### Ask AI
*After class:* ask an AI assistant why dropout is called "training an ensemble of smaller networks inside one big network" — get a different analogy than the sports-team one from class.

## Fix 3 — Batch Normalization

**Analogy:** a chef who recalibrates the kitchen scale before every use vs one that silently drifts, forcing constant guesswork.

- Problem: as early layers update, later layers' *input distribution* keeps shifting -> moving target
- Fix: normalize each layer's input per mini-batch

$$\mu_B = \frac{1}{n}\sum z_i \qquad \sigma_B^2 = \frac{1}{n}\sum(z_i-\mu_B)^2 \qquad \hat{z}_i = \frac{z_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}} \qquad y_i = \gamma\hat{z}_i + \beta$$

- $n$ = batch size, $z_i$ = pre-activation value, $\epsilon$ = tiny constant (avoids /0)
- $\gamma, \beta$ = **learnable** scale/shift — let the network undo strict normalization if that's better for it

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/241/991/original/ChatGPT_Image_Sep_23__2026__05_46_41_PM.png?1790165810" height=500>

### Quiz 5

Why are $\gamma$ and $\beta$ learnable instead of forcing every normalized output to always have mean 0 and variance 1?

- A) They only matter at test time

- B) Strict mean-0/variance-1 normalization can limit what a layer can represent; learnable $\gamma,\beta$ let the network adjust the scale and shift while keeping the benefits of normalization

- C) They replace the activation function

- D) They are fixed constants and never updated

**Answer: B)**

> **Explanation:** Normalization keeps activations stable, but forcing them to always stay at mean 0 and variance 1 can be too restrictive. Learnable $\gamma$ and $\beta$ allow the network to **scale and shift** the normalized values when needed, so the layer can learn the most useful distribution for the task.

### Ask AI
*After class:* ask an AI assistant to explain **"internal covariate shift"** — the problem BatchNorm was built to fix — and what happens in a very deep (~100 layer) network if it's left unaddressed.

## Fix 4 — Early Stopping

**Analogy:** a cake left in the oven past the right time doesn't improve — it just dries out.

- Meera trains longer, expecting accuracy to keep climbing
- Instead: val accuracy peaks, then *declines* while train accuracy keeps rising -> back into overfitting
- Fix: after every epoch, check val performance, save the best version so far; stop once val performance stalls for a set number of epochs (**patience**)

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/241/993/original/ChatGPT_Image_Sep_23__2026__05_48_49_PM.png?1790165940" height=500>

## AI in the Loop

- Every fix today = a hyperparameter someone has to choose ($\lambda$, $r$, BatchNorm placement, patience)
- Industry teams increasingly automate this:
  - **Automated hyperparameter search** (e.g. Bayesian optimization tools) tries combinations systematically, guided by validation results
  - **AI assistants** can read a training-curve description ("train 90%, val 68%, gap widened after epoch 15") and suggest which fix is likely to help — a first-pass debugging partner
- Doesn't replace today's understanding — someone still needs to interpret and sanity-check what the automation suggests



In [ ]:
# Interactive Playground: Dropout on a hidden layer
# Move the slider to change dropout rate r and see which neurons drop for a fresh training step.

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

np.random.seed(7)
NUM_NEURONS = 64
GRID_SHAPE = (8, 8)
base_activation = np.random.uniform(0.4, 1.0, NUM_NEURONS)

def show_dropout(dropout_rate=0.3):
    mask = (np.random.rand(NUM_NEURONS) > dropout_rate).astype(float)
    scaled = (base_activation * mask) / (1 - dropout_rate + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
    axes[0].imshow(base_activation.reshape(GRID_SHAPE), cmap="Greens", vmin=0, vmax=1.3)
    axes[0].set_title("No dropout")
    axes[0].set_xticks([]); axes[0].set_yticks([])

    axes[1].imshow(scaled.reshape(GRID_SHAPE), cmap="Greens", vmin=0, vmax=1.3)
    axes[1].set_title(f"r = {dropout_rate:.2f}  ({int(round(dropout_rate*NUM_NEURONS))}/{NUM_NEURONS} dropped)")
    axes[1].set_xticks([]); axes[1].set_yticks([])

    plt.tight_layout()
    plt.show()

interact(show_dropout, dropout_rate=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.3, description="dropout r"));

interactive(children=(FloatSlider(value=0.3, description='dropout r', max=0.8, step=0.05), Output()), _dom_cla…

## Wrap-Up

- Overfitting = train/val gap from memorizing noise, not pattern
- Target encoding + standardization prep data for stable NN training
- Sigmoid + Binary Cross-Entropy for probability-vs-binary-label problems
- **L2 Reg**: weight decay via $\lambda$
- **Dropout**: random masking via rate $r$, scaled so train/test stay consistent
- **BatchNorm**: normalize per batch, restore flexibility via $\gamma, \beta$
- **Early Stopping**: halt training at the validation peak
- AI/automation extends, but doesn't replace, this diagnostic process